# Dijkstra算法详解及Python实现

## 算法概述

Dijkstra算法是由荷兰计算机科学家Edsger W. Dijkstra于1956年提出的**单源最短路径算法**，用于在带权有向图或无向图中找到从一个源点到所有其他顶点的最短路径。

### 核心思想
- **贪心策略**：每次选择当前已知最短路径的顶点
- **逐步扩展**：从源点开始，逐步向外扩展到所有顶点
- **权重非负**：要求图中所有边的权重为非负值

## 算法步骤

1. **初始化**
   - 创建距离字典，源点距离设为0，其他顶点设为无穷大
   - 创建优先队列（最小堆），包含所有顶点
   - 创建前驱节点字典，记录最短路径

2. **主循环**
   - 从优先队列中取出距离最小的顶点u
   - 遍历u的所有邻接顶点v
   - 计算经过u到v的距离
   - 如果新距离小于当前已知距离，更新距离和前驱节点

3. **终止条件**
   - 所有顶点都被处理，或
   - 目标顶点被处理（如果只关心到特定顶点的路径）

## Python实现

### 基础版本（使用列表）

In [ ]:
import sys

class Graph:
    def __init__(self, vertices):
        self.V = vertices
        self.graph = [[0 for _ in range(vertices)] for _ in range(vertices)]
    
    def dijkstra(self, src):
        # 初始化距离数组
        dist = [sys.maxsize] * self.V
        dist[src] = 0
        visited = [False] * self.V
        
        for _ in range(self.V):
            # 找到未访问顶点中距离最小的
            u = self._min_distance(dist, visited)
            visited[u] = True
            
            # 更新邻接顶点的距离
            for v in range(self.V):
                if (self.graph[u][v] > 0 and 
                    not visited[v] and 
                    dist[v] > dist[u] + self.graph[u][v]):
                    dist[v] = dist[u] + self.graph[u][v]
        
        self._print_solution(dist)
    
    def _min_distance(self, dist, visited):
        min_val = sys.maxsize
        min_index = -1
        
        for v in range(self.V):
            if dist[v] < min_val and not visited[v]:
                min_val = dist[v]
                min_index = v
                
        return min_index
    
    def _print_solution(self, dist):
        print("顶点\t距离源点的距离")
        for node in range(self.V):
            print(f"{node}\t{dist[node]}")

# 使用示例
g = Graph(9)
g.graph = [
    [0, 4, 0, 0, 0, 0, 0, 8, 0],
    [4, 0, 8, 0, 0, 0, 0, 11, 0],
    [0, 8, 0, 7, 0, 4, 0, 0, 2],
    [0, 0, 7, 0, 9, 14, 0, 0, 0],
    [0, 0, 0, 9, 0, 10, 0, 0, 0],
    [0, 0, 4, 14, 10, 0, 2, 0, 0],
    [0, 0, 0, 0, 0, 2, 0, 1, 6],
    [8, 11, 0, 0, 0, 0, 1, 0, 7],
    [0, 0, 2, 0, 0, 0, 6, 7, 0]
]

g.dijkstra(0)


### 优化版本（使用优先队列）

In [ ]:
import heapq
from collections import defaultdict

def dijkstra_optimized(graph, start):
    """
    使用优先队列优化的Dijkstra算法
    
    参数:
    graph: 邻接表表示的图 {节点: {邻居: 权重}}
    start: 起始节点
    
    返回:
    distances: 从起点到各节点的最短距离
    previous: 最短路径的前驱节点
    """
    # 初始化距离字典
    distances = {node: float('infinity') for node in graph}
    distances[start] = 0
    
    # 优先队列: (距离, 节点)
    priority_queue = [(0, start)]
    previous = {node: None for node in graph}
    
    while priority_queue:
        current_distance, current_node = heapq.heappop(priority_queue)
        
        # 如果找到更短的路径已经处理过该节点，跳过
        if current_distance > distances[current_node]:
            continue
            
        # 遍历邻居节点
        for neighbor, weight in graph[current_node].items():
            distance = current_distance + weight
            
            # 如果找到更短的路径
            if distance < distances[neighbor]:
                distances[neighbor] = distance
                previous[neighbor] = current_node
                heapq.heappush(priority_queue, (distance, neighbor))
    
    return distances, previous

def reconstruct_path(previous, start, end):
    """重构从起点到终点的最短路径"""
    path = []
    current = end
    
    while current is not None:
        path.append(current)
        current = previous[current]
    
    path.reverse()
    return path if path[0] == start else []

# 使用示例
graph = {
    'A': {'B': 1, 'C': 4},
    'B': {'A': 1, 'C': 2, 'D': 5},
    'C': {'A': 4, 'B': 2, 'D': 1},
    'D': {'B': 5, 'C': 1}
}

distances, previous = dijkstra_optimized(graph, 'A')
print("最短距离:", distances)

path = reconstruct_path(previous, 'A', 'D')
print("最短路径:", path)


### 完整应用示例

In [ ]:
import heapq
from typing import Dict, List, Tuple

class DijkstraRouter:
    """基于Dijkstra算法的路径规划器"""
    
    def __init__(self, graph: Dict[str, Dict[str, float]]):
        self.graph = graph
    
    def find_shortest_path(self, start: str, end: str) -> Tuple[float, List[str]]:
        """找到从起点到终点的最短路径和距离"""
        distances = {node: float('infinity') for node in self.graph}
        distances[start] = 0
        previous = {node: None for node in self.graph}
        
        priority_queue = [(0, start)]
        
        while priority_queue:
            current_distance, current_node = heapq.heappop(priority_queue)
            
            if current_distance > distances[current_node]:
                continue
                
            if current_node == end:
                break
                
            for neighbor, weight in self.graph.get(current_node, {}).items():
                distance = current_distance + weight
                
                if distance < distances[neighbor]:
                    distances[neighbor] = distance
                    previous[neighbor] = current_node
                    heapq.heappush(priority_queue, (distance, neighbor))
        
        path = self._reconstruct_path(previous, start, end)
        return distances[end], path
    
    def _reconstruct_path(self, previous: Dict[str, str], start: str, end: str) -> List[str]:
        """重构路径"""
        path = []
        current = end
        
        while current is not None:
            path.append(current)
            current = previous[current]
        
        path.reverse()
        return path if path and path[0] == start else []
    
    def find_all_shortest_paths(self, start: str) -> Dict[str, Tuple[float, List[str]]]:
        """找到从起点到所有其他节点的最短路径"""
        results = {}
        
        for end_node in self.graph:
            if end_node != start:
                distance, path = self.find_shortest_path(start, end_node)
                results[end_node] = (distance, path)
        
        return results

# 实际应用示例：城市导航
city_map = {
    '家': {'超市': 5, '学校': 10, '公园': 8},
    '超市': {'家': 5, '学校': 3, '商场': 7},
    '学校': {'家': 10, '超市': 3, '公园': 4, '体育馆': 6},
    '公园': {'家': 8, '学校': 4, '体育馆': 2},
    '商场': {'超市': 7, '体育馆': 5},
    '体育馆': {'学校': 6, '公园': 2, '商场': 5}
}

router = DijkstraRouter(city_map)

# 找到从家到体育馆的最短路径
distance, path = router.find_shortest_path('家', '体育馆')
print(f"从家到体育馆的最短距离: {distance}")
print(f"路径: {' -> '.join(path)}")

print("\n从家到所有地点的最短路径:")
all_paths = router.find_all_shortest_paths('家')
for destination, (dist, route) in all_paths.items():
    print(f"到 {destination}: 距离={dist}, 路径={' -> '.join(route)}")


## 算法复杂度分析

- **时间复杂度**：
  - 基础版本（使用列表）：O(V²)
  - 优化版本（使用优先队列）：O((V+E)logV)
  
- **空间复杂度**：O(V)

## 应用场景

1. **网络路由**：IP网络中的最短路径路由
2. **地图导航**：GPS导航系统
3. **社交网络**：查找人际关系的最短路径
4. **游戏开发**：AI寻路算法
5. **交通规划**：公共交通路线优化

## 注意事项

1. **负权边**：Dijkstra算法不能处理负权边，需要使用Bellman-Ford算法
2. **性能考虑**：对于稀疏图，优先队列版本性能更好
3. **内存优化**：对于大型图，可以考虑使用双向Dijkstra算法

这个算法在现实世界中有着广泛的应用，是图论中最基础且重要的算法之一。